# Spaceship Titanic TC1 Human Baseline

This notebook defines the notebook-backed TC1 human baseline. It mirrors a plausible manual preprocessing pass and can be rerun as-is; committed baseline outputs and evidence are recorded in the JSON/provenance artifacts.


## Load Data


In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path.home() / 'Downloads' / 'spaceship-titanic'
required_files = ['train.csv', 'test.csv', 'sample_submission.csv']
missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(f'Missing required files under {DATA_DIR}: {missing_files}')

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print('train shape:', train.shape)
print('test shape:', test.shape)
print('sample_submission shape:', sample_submission.shape)


## Split Cabin Into Structured Parts


In [ ]:
for frame in (train, test):
    frame[['Deck', 'CabinNum', 'Side']] = frame['Cabin'].str.split('/', expand=True)


## Split PassengerId Into Group Features


In [ ]:
for frame in (train, test):
    frame[['PassengerGroup', 'PassengerNum']] = frame['PassengerId'].str.split('_', expand=True)


## Align Spending With CryoSleep


In [ ]:
spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
for frame in (train, test):
    mask = frame['CryoSleep'] == True
    frame.loc[mask, spend_cols] = 0


## Fill Spending Columns With Zero


In [ ]:
for frame in (train, test):
    frame[spend_cols] = frame[spend_cols].fillna(0)


## Create TotalSpend And NoSpending


In [ ]:
for frame in (train, test):
    frame['TotalSpend'] = frame[spend_cols].sum(axis=1)
    frame['NoSpending'] = (frame['TotalSpend'] == 0).astype(int)


## Impute Age


In [ ]:
train['Age'] = train['Age'].fillna(train['Age'].median())
test['Age'] = test['Age'].fillna(train['Age'].median())


## Impute Categorical Flags


In [ ]:
for column in ['HomePlanet', 'Destination', 'CryoSleep', 'VIP']:
    mode_value = train[column].mode().iloc[0]
    train[column] = train[column].fillna(mode_value)
    test[column] = test[column].fillna(mode_value)


## Create GroupSize


In [ ]:
for frame in (train, test):
    frame['GroupSize'] = frame.groupby('PassengerGroup')['PassengerId'].transform('count')


## Encode Nominal Columns


In [ ]:
encoding_maps = {}
for column in ['HomePlanet', 'Destination', 'Deck', 'Side', 'VIP', 'CryoSleep']:
    combined = pd.concat([train[column], test[column]], axis=0).astype(str)
    categories = list(pd.unique(combined))
    mapping = {value: idx for idx, value in enumerate(categories)}
    encoding_maps[column] = mapping
    train[column] = train[column].astype(str).map(mapping).astype(int)
    test[column] = test[column].astype(str).map(mapping).astype(int)

encoding_maps


## Drop Raw Identifier Columns


In [ ]:
train = train.drop(columns=['PassengerId', 'Name', 'Cabin'])
test = test.drop(columns=['PassengerId', 'Name', 'Cabin'])

print('processed train shape:', train.shape)
print('processed test shape:', test.shape)
print('submission rows match test rows:', len(sample_submission) == len(test))
train.head()
